In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, classification_report, confusion_matrix
import pickle as pkl

In [ ]:
df = pd.read_csv('customer_booking.csv', encoding='latin-1')
df

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df.sales_channel.value_counts()

In [ ]:
df.trip_type.value_counts()

In [ ]:
df.flight_day.value_counts()

In [ ]:
df.route.value_counts()

In [ ]:
df.booking_origin.value_counts()

In [ ]:
days = {
    'Mon':1,
    'Tue':2,
    'Wed':3,
    'Thu':4,
    'Fri':5,
    'Sat':6,
    'Sun':7
}
df['flight_day'] = df['flight_day'].map(days)

In [ ]:
df

In [ ]:
correlation = df.corr()
correlation

In [ ]:
sns.heatmap(correlation, cmap='inferno')

In [ ]:
sns.boxplot(x=df['num_passengers'])

In [ ]:
sns.boxplot(x=df['flight_day'], color='green')

In [ ]:
keys = df.num_passengers.value_counts()
keys

In [ ]:
num_pass_keys = []
num_pass_values = []
for i, j in keys.items():
    num_pass_keys.append(i)
    num_pass_values.append(j)
print(num_pass_keys, num_pass_values)

In [ ]:
plt.pie(num_pass_values, labels=num_pass_keys, explode=[0.1, 0.1, 0.1, 0.1, 0.3, 0.5, 0.6, 0.9, 0], startangle=140)
plt.subplots_adjust(left=0.1, right=0.9, top=0.9, bottom=0.1)
plt.axis('equal')
plt.legend()
plt.show()

In [ ]:
sns.boxplot(x=df['sales_channel'], y=df['num_passengers'])
plt.subplots_adjust(left=0.1, right=0.9, top=0.9, bottom=0.1)
plt.axis('equal')
plt.legend()
plt.show()

In [ ]:
keys = df.sales_channel.value_counts()
keys

In [ ]:
num_pass_keys = []
num_pass_values = []
for i, j in keys.items():
    num_pass_keys.append(i)
    num_pass_values.append(j)
print(num_pass_keys, num_pass_values)

In [ ]:
plt.pie(num_pass_values, labels=num_pass_keys, explode=[0.1, 0.1], colors=['violet','black'], startangle=140, autopct='%1.1f%%')
plt.subplots_adjust(left=0.1, right=0.9, top=0.9, bottom=0.1)
plt.axis('equal')
plt.legend()
plt.show()

In [ ]:
keys = df.trip_type.value_counts()
keys

In [ ]:
num_pass_keys = []
num_pass_values = []
for i, j in keys.items():
    num_pass_keys.append(i)
    num_pass_values.append(j)
print(num_pass_keys, num_pass_values)

In [ ]:
plt.pie(num_pass_values, labels=num_pass_keys, explode=[0.1, 0.1, 0.5], startangle=140, autopct='%1.1f%%')
plt.subplots_adjust(left=0.1, right=0.9, top=0.9, bottom=0.1)
plt.axis('equal')
plt.legend()
plt.show()

In [ ]:
(df['booking_origin'].value_counts())

In [ ]:
df.route.value_counts()

In [ ]:
df

In [ ]:
sales_channel_le = LabelEncoder()
trip_type_le = LabelEncoder()

In [ ]:
df['sales_channel'] = sales_channel_le.fit_transform(df['sales_channel'])
df['trip_type'] = trip_type_le.fit_transform(df['trip_type'])

In [ ]:
df

In [ ]:
sales_channel_le.classes_

In [ ]:
trip_type_le.classes_

In [ ]:
df.booking_complete.value_counts()

In [ ]:
encoder = OneHotEncoder(sparse=False, handle_unknown='ignore')

In [ ]:
df[['route', 'booking_origin']]

In [ ]:
route_booking = encoder.fit_transform(df[['booking_origin']])
route_booking

In [ ]:
encoder.categories_

In [ ]:
feature_names = []
for feature in encoder.categories_:
    feature_names.extend(feature)

In [ ]:
route_booking_df = pd.DataFrame(route_booking, columns=feature_names)
route_booking_df

In [ ]:
features = df.drop(['route', 'booking_origin', 'booking_complete'], axis =1)
features

In [ ]:
features = pd.concat([features, route_booking_df], axis = 1)
features

In [ ]:
booking_target = df[['booking_complete']]
booking_target

In [ ]:
booking_target.value_counts()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(features, booking_target, test_size=0.2, random_state=42)

In [ ]:
Y_train = np.ravel(y_train)

In [ ]:
algorithms = {
    'Decision Tree' : {
        'model': DecisionTreeClassifier(criterion='entropy'),
        'param':{}
    },
    
    'Random Forest' : {
        'model': RandomForestClassifier(),
        'param':{}
    },
    
    'Adaabooster' : {
        'model': AdaBoostClassifier(),
        'param':{}
    },
    
    'MultinomialNB' : {
        'model': MultinomialNB(),
        'param':{}
    },
    
}

In [ ]:
algorithms.keys()

In [ ]:
scores =[]
for model_name,mp in algorithms.items():
    print(mp)
    print(model_name)
    clf =  GridSearchCV(mp['model'], mp['param'], cv=5, return_train_score=False)
    clf.fit(X_train, Y_train)
    Test_score = clf.score(X_test,y_test)
    print("Completed", mp['model'])
    scores.append({
        'model': model_name,
        'best_score': clf.best_score_,
        'best_params': clf.best_params_,
        'Test score': Test_score    
    })

In [ ]:
scores

We are selecting the Adaboost algorithm over Random Forest based on its slightly higher test score. While both algorithms exhibit equally high scores in both test and validation sets, we prioritize the test score because it reflects the algorithm's performance on unseen data, which is more critical compared to the training data's impact.

# Training Model

In [ ]:
base_estimator = DecisionTreeClassifier(max_depth=1)
ada_model = AdaBoostClassifier(base_estimator=base_estimator, n_estimators= 200, learning_rate= 0.1,algorithm = 'SAMME.R')
ada_model

In [ ]:
ada_model.fit(features, booking_target['booking_complete'])

In [ ]:
ada_model.score(X_test,y_test)

In [ ]:
ada_model.score(X_train,y_train)

# Metrics of Model

In [ ]:
y_pred = ada_model.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm=confusion_matrix(y_test,y_pred)
cm

In [ ]:
sns.heatmap(cm, annot=True, cmap='cividis')

In [ ]:
with open('model/ada_british_airways_model.pkl', 'wb') as file:
    pkl.dump(ada_model, file)